# 04: GiT flow-matching training

Read anchor sections and morphology features, train the histology-guided joint flow-matching model, and save fitted checkpoints.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/03_3d").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "00029_native_v2")
OUT = ROOT / "outputs/03_3d" / RUN_ID
DEVICE = os.environ.get(
    "SPAGC_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu"
)
SEED = 29
MODE = "path"
USE_CELLTYPE = True
LABEL_KEY = "cell_class"
SPATIAL_KEY = "spatial_st_corrected"
Z_KEY = "z_um"
FRAME = "00029__g0_registered"
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
    raise ValueError("SPAGC_RUN_ID must be a safe directory name")
DATA = ROOT / "data/3d/00029_g0_native"
ANCHORS = DATA / "st/anchors_celltype_domain_niche"
CELLTYPE_LABELS = DATA / "cell_type_labels.csv"
ARTIFACTS = ROOT / "outputs/03_3d" / RUN_ID
FEATURES = ARTIFACTS / "uni2_features.h5"
PATH_CACHE = ARTIFACTS / "morphology_paths.h5"
MODEL_DIR = ARTIFACTS / "model"
if MODE not in ("baseline", "uot", "path"):
    raise ValueError("Unsupported run mode or model mode.")
print({"model_mode": MODE, "output": str(OUT)})
SECTION_RANGE = None
ROI_UM = None
MAX_CELLS_PER_ANCHOR = None
BATCH_SIZE = 128
N_PAIRS = 50000
TOP_K = 64
EPOCHS = 100
MODEL_PATCH_SIZE = 8
MODEL_HIDDEN_SIZE = 256
MODEL_DEPTH = 6
MODEL_HEADS = 8
NUCLEUS_FEATURES = ARTIFACTS / "aligned_nucleus_features.h5"
USE_NUCLEUS_PATH = False
NUCLEUS_WEIGHT = 1.0
if USE_NUCLEUS_PATH and MODE != "path":
    raise ValueError(
        "The optional supplied-nucleus experiment requires full mode and morphology paths."
    )
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from spagc._io import require_new
from spagc.reconstruction.core import DeepSpatial
from spagc.reconstruction.histology import HistologyConfig
from spagc.reconstruction.inputs import load_anchors

Read the real anchor sections and require morphology features for the declared histology mode.

Read measured anchor sections, retain their expression and physical coordinates, and apply only the explicitly selected spatial or cell subset.

In [ ]:
anchors = load_anchors(
    ANCHORS,
    require_labels=USE_CELLTYPE,
    labels_path=CELLTYPE_LABELS,
    section_range=SECTION_RANGE,
    roi_um=ROI_UM,
    max_cells_per_anchor=MAX_CELLS_PER_ANCHOR,
    seed=SEED,
    label_key=LABEL_KEY,
    spatial_key=SPATIAL_KEY,
    z_key=Z_KEY,
)
feature_path = FEATURES
if MODE != "baseline" and (not feature_path.is_file()):
    raise FileNotFoundError(f"Generate real morphology features first: {feature_path}")

## 2. Specify morphology-aware supervision

Select baseline, morphology-aware pairing, or pairing plus morphology-guided paths. Nuclear-anatomy penalties remain an explicitly enabled option.

Specify which morphology-guided pairing and spatial-path terms enter the reconstruction.

In [ ]:
config = HistologyConfig(
    use_histology=MODE != "baseline",
    use_morphology_uot=MODE != "baseline",
    use_morphology_path=MODE == "path",
    use_nucleus_path=USE_NUCLEUS_PATH,
    nucleus_feature_path=str(NUCLEUS_FEATURES) if USE_NUCLEUS_PATH else None,
    nucleus_weight=NUCLEUS_WEIGHT if USE_NUCLEUS_PATH else 0.0,
    feature_path=str(feature_path) if MODE != "baseline" else None,
    path_cache=str(PATH_CACHE) if MODE == "path" else None,
    coordinate_frame=FRAME if MODE != "baseline" else None,
    seed=SEED,
    candidate_radius_um=100,
    candidate_spacing_um=25,
    allow_linear_fallback=False,
)

## 3. Build transport-guided training pairs

Prepare the measured anchors, transport couplings and conditional flow-matching training samples. Retain the supplied cell-type vocabulary when enabled.

In [ ]:
model = DeepSpatial()
model.setup_data(
    anchors,
    spatial_key=SPATIAL_KEY,
    z_key=Z_KEY,
    label_key=LABEL_KEY,
    use_celltype=USE_CELLTYPE,
    histology=config,
    batch_size=BATCH_SIZE,
    num_workers=0,
    n_samples_base=N_PAIRS,
    uot_solver="sparse_topk",
    uot_top_k=TOP_K,
)

## 4. Construct the velocity-prediction model

Build the inherited joint spatial/molecular transformer with the declared architecture and trajectory-integration settings.

In [ ]:
model.build_model(
    patch_size=MODEL_PATCH_SIZE,
    hidden_size=MODEL_HIDDEN_SIZE,
    depth=MODEL_DEPTH,
    num_heads=MODEL_HEADS,
    sampling_method="dopri5",
)
print(
    {
        "pairs": len(model.dataset),
        "gene_dim": model.gene_dim,
        "classes": model.num_classes,
    }
)

## 5. Train and save model checkpoints

Optimize the joint velocity-regression objective and save checkpoints, model configuration and training artifacts. Earlier output directories are not overwritten.

In [ ]:
save_dir = MODEL_DIR
require_new(save_dir)
model.fit(
    max_epochs=EPOCHS,
    save_dir=str(save_dir),
    accelerator="cpu" if DEVICE == "cpu" else "gpu",
    devices=1 if DEVICE == "cpu" else [int(DEVICE.split(":")[-1])],
    save_ckpt=True,
)
checkpoints = sorted(save_dir.glob("*.ckpt"))
assert checkpoints and (save_dir / "config.json").is_file()